# NUS DSA4211 — High-Dimensional Statistical Analysis
## End-to-end Kaggle case study: Leukemia Gene Expression (CuMiDa GSE9476)

This notebook is a **tutorial + case study** for the main ideas in high-dimensional statistical analysis.

We use the public Kaggle dataset **Leukemia gene expression – CuMiDa (GSE9476)**:

- **64 samples**
- **22,283 gene-expression features**
- **5 classes**
- CSV shape of about **64 × 22,285** after including the sample identifier and target label.

This is an extreme example of the central regime

\[
p \gg n.
\]

With roughly 22,283 predictors and 64 observations,

\[
\frac{p}{n}\approx348.
\]

So the data contain roughly **348 candidate gene features per sample**.

---

## Topics illustrated

1. Why ordinary least squares becomes ill-posed for \(p>n\)
2. Matrix rank, singular Gram matrices, and non-identifiability
3. Curse of dimensionality and distance concentration
4. PCA and low-rank structure
5. Sliced Inverse Regression (regularized pedagogical version)
6. SIS-style marginal screening
7. Ridge, LASSO, and Elastic Net
8. Sparse coefficient paths
9. Leakage-safe pipelines
10. Repeated stratified cross-validation
11. Nested cross-validation for model selection
12. Logistic regression, k-NN, trees, random forests, and MLPs
13. Thousands of simultaneous tests
14. Bonferroni, Holm, and Benjamini–Hochberg FDR
15. Effect sizes
16. K-means and hierarchical clustering
17. Prediction vs inference
18. Sample splitting for simple post-selection inference
19. Why HMMs are not appropriate for this specific Kaggle dataset
20. A separate synthetic HMM demonstration
21. Computational complexity
22. Exercises and worked solutions

> **Statistical caution:** With only 64 observations, validation uncertainty is substantial. This is an educational statistical-learning case study, not a clinical diagnostic system.

## 1. Dataset source

Kaggle slug:

```text
brunogrisci/leukemia-gene-expression-cumida
```

The dataset is based on **GSE9476** and was curated in the CuMiDa project.

References:

- Kaggle: https://www.kaggle.com/datasets/brunogrisci/leukemia-gene-expression-cumida
- Feltes et al. (2019), *CuMiDa: An Extensively Curated Microarray Database for Benchmarking and Testing of Machine Learning Approaches in Cancer Research*
- GEO accession: GSE9476

The first download cell uses `kagglehub`. If local Kaggle access is unavailable, set `LOCAL_CSV` to a manually downloaded `Leukemia_GSE9476.csv`.

In [1]:
# Run once if the environment does not already contain these packages.
%pip install -q kagglehub bokeh scikit-learn statsmodels scipy pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Optional, Tuple
import warnings

import numpy as np
import pandas as pd
from scipy import stats
from scipy.spatial.distance import pdist

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.model_selection import (
    GridSearchCV,
    RepeatedStratifiedKFold,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from statsmodels.stats.multitest import multipletests

from bokeh.io import output_notebook, show
from bokeh.layouts import row
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.palettes import Category10
from bokeh.plotting import figure
from bokeh.transform import factor_cmap

output_notebook()
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

Loading BokehJS ...

## 2. Configuration

`FAST_MODE=True` keeps repeated-CV experiments lighter while preserving the workflow. Set it to `False` for more repetitions.

A recurring principle in this notebook is that **every learned preprocessing operation must be fitted inside the training fold** when reporting predictive performance.

In [3]:
@dataclass(frozen=True)
class CaseStudyConfig:
    kaggle_slug: str = 'brunogrisci/leukemia-gene-expression-cumida'
    csv_name: str = 'Leukemia_GSE9476.csv'
    target_col: str = 'type'
    sample_col: str = 'samples'
    screening_k: int = 200
    random_state: int = 42
    fast_mode: bool = True

CFG = CaseStudyConfig()
CFG

CaseStudyConfig(kaggle_slug='brunogrisci/leukemia-gene-expression-cumida', csv_name='Leukemia_GSE9476.csv', target_col='type', sample_col='samples', screening_k=200, random_state=42, fast_mode=True)

## 3. Download and load the data

The loader searches recursively for the CSV because KaggleHub's cache directory can vary by environment.

In [4]:
LOCAL_CSV: Optional[str] = None
# Example:
# LOCAL_CSV = '/home/you/data/Leukemia_GSE9476.csv'

if LOCAL_CSV is None:
    import kagglehub
    dataset_dir = Path(kagglehub.dataset_download(CFG.kaggle_slug))
    candidates = list(dataset_dir.rglob(CFG.csv_name))
    if not candidates:
        candidates = list(dataset_dir.rglob('*.csv'))
    if not candidates:
        raise FileNotFoundError(f'No CSV found below {dataset_dir}')
    csv_path = candidates[0]
else:
    csv_path = Path(LOCAL_CSV)

print('Using:', csv_path)
df = pd.read_csv(csv_path)
print('Shape:', df.shape)
display(df.iloc[:5, :12])

Using: /home/anirban/.cache/kagglehub/datasets/brunogrisci/leukemia-gene-expression-cumida/versions/1/Leukemia_GSE9476.csv
Shape: (64, 22285)


,samples,type,1007_s_at,1053_at,117_at,121_at,1255_g_at,1294_at,1316_at,1320_at,1405_i_at,1431_at
0,1,Bone_Marrow_CD34,7.745245,7.811210,6.477916,8.841506,4.546941,7.957714,5.344999,4.673364,4.664924,4.069624
1,12,Bone_Marrow_CD34,8.087252,7.240673,8.584648,8.983571,4.548934,8.011652,5.579647,4.828184,5.171835,4.299875
2,13,Bone_Marrow_CD34,7.792056,7.549368,11.053504,8.909703,4.549328,8.237099,5.406489,4.615572,4.775709,4.148363
3,14,Bone_Marrow_CD34,7.767265,7.094460,11.816433,8.994654,4.697018,8.283412,5.582195,4.903684,4.829844,4.075494
4,15,Bone_Marrow_CD34,8.010117,7.405281,6.656049,9.050682,4.514986,8.377046,5.493713,4.860754,5.245049,4.052077


## 4. Validate the schema and define \(X\) and \(y\)

Published descriptions report 64 samples, five classes, a sample identifier named `samples`, and a target named `type`.

We validate the actual file rather than silently trusting documentation.

In [5]:
assert CFG.target_col in df.columns, (
    f"Expected target column {CFG.target_col!r}; first columns are {df.columns[:10].tolist()}"
)

exclude = [CFG.target_col]
if CFG.sample_col in df.columns:
    exclude.append(CFG.sample_col)

feature_cols = [c for c in df.columns if c not in exclude]
X = df[feature_cols].apply(pd.to_numeric, errors='coerce')
y = df[CFG.target_col].astype(str)

if X.isna().any().any():
    missing = int(X.isna().sum().sum())
    print(f'Warning: {missing} missing/non-numeric cells. Median-imputing as a defensive fallback.')
    X = X.fillna(X.median())

n, p = X.shape

print(f'n samples        = {n:,}')
print(f'p gene features = {p:,}')
print(f'p / n           = {p/n:,.1f}')
print(f'classes         = {y.nunique()}')
print()
print(y.value_counts())

n samples        = 64
p gene features = 22,283
p / n           = 348.2
classes         = 5

type
AML                 26
Bone_Marrow         10
PB                  10
PBSC_CD34           10
Bone_Marrow_CD34     8
Name: count, dtype: int64


# Part I — Why this problem is genuinely high-dimensional

## 5. The rank problem

For a linear model

\[
y=X\beta+\varepsilon,
\qquad X\in\mathbb{R}^{n\times p},
\]

we always have

\[
\operatorname{rank}(X)\le\min(n,p).
\]

When \(p>n\), \(X\) cannot have column rank \(p\). Therefore

\[
X^\top X
\]

is singular and the classical OLS inverse

\[
(X^\top X)^{-1}
\]

does not exist.

After centering 64 samples, the rank is at most 63 because the centered rows sum to zero.

In [6]:
summary_df = pd.DataFrame({
    'quantity': ['samples n', 'features p', 'p/n', 'max centered rank'],
    'value': [n, p, p / n, n - 1],
})
display(summary_df)

,quantity,value
0,samples n,64.000000
1,features p,22283.000000
2,p/n,348.171875
3,max centered rank,63.000000


## 6. Class imbalance

AML is the largest group. Therefore accuracy alone is not enough.

Later we report:

- ordinary accuracy,
- balanced accuracy,
- macro-F1.

Macro-F1 gives each class equal weight.

In [7]:
class_counts = y.value_counts().sort_index()
classes = class_counts.index.tolist()

source = ColumnDataSource(pd.DataFrame({
    'class': classes,
    'count': class_counts.values,
}))

p_class = figure(
    x_range=classes,
    height=350,
    width=850,
    title='Class distribution',
    toolbar_location=None,
)
p_class.vbar(x='class', top='count', width=0.7, source=source)
p_class.xaxis.major_label_orientation = 0.7
p_class.yaxis.axis_label = 'Samples'
show(p_class)

## 7. Empirical rank deficiency

We do **not** build the full \(22{,}283\times22{,}283\) Gram matrix because that is wasteful.

Instead, use only 200 genes. Even then,

\[
200>64,
\]

so the centered design is necessarily rank deficient.

In [8]:
p_demo = min(200, p)
X_demo = StandardScaler(with_std=False).fit_transform(X.iloc[:, :p_demo])

rank_demo = np.linalg.matrix_rank(X_demo)
gram_demo = X_demo.T @ X_demo
eigvals_demo = np.linalg.eigvalsh(gram_demo)

print('Demo matrix shape:', X_demo.shape)
print('Rank:', rank_demo)
print('Rank deficiency:', p_demo - rank_demo)
print(f'Smallest eigenvalue of X^T X: {eigvals_demo.min():.3e}')
print('Near-zero eigenvalues (<1e-8):', int((eigvals_demo < 1e-8).sum()))

Demo matrix shape: (64, 200)
Rank: 63
Rank deficiency: 137
Smallest eigenvalue of X^T X: -1.254e-13
Near-zero eigenvalues (<1e-8): 137


### What high-dimensional methods add

The data alone cannot identify 22k unrestricted coefficients from 64 observations.

We therefore impose structure:

- **Ridge:** coefficients should collectively have small \(L_2\) norm.
- **LASSO:** the true model is sparse.
- **PCA:** the useful signal lies in a low-rank subspace.
- **SIS:** only a small candidate set needs expensive modelling.
- **Trees:** prediction can be represented by a small hierarchy of splits.
- **Multiple-testing procedures:** error rates must be controlled over thousands of hypotheses.

## 8. Curse of dimensionality: distance concentration

The leukemia data contain strong biological structure, so a pure geometric effect is clearer with a synthetic experiment.

For random points in increasing dimension, Euclidean distances tend to concentrate.

We track

\[
CV(d)=\frac{sd(d)}{mean(d)}
\]

and the intuitive relative contrast

\[
\frac{d_{max}-d_{min}}{d_{min}}.
\]

In [9]:
dims = [2, 5, 10, 20, 50, 100, 250, 500, 1000]
rows = []

for d in dims:
    Z = rng.normal(size=(250, d))
    distances = pdist(Z, metric='euclidean')
    rows.append({
        'dimension': d,
        'distance_cv': distances.std() / distances.mean(),
        'relative_contrast': (distances.max() - distances.min()) / distances.min(),
    })

distance_df = pd.DataFrame(rows)
display(distance_df)

,dimension,distance_cv,relative_contrast
0,2,0.530555,503.673260
1,5,0.321726,24.656807
2,10,0.224763,7.552655
3,20,0.160020,2.582433
4,50,0.092662,1.221230
5,100,0.069567,0.746278
6,250,0.043764,0.414883
7,500,0.030734,0.283227
8,1000,0.021952,0.223313


In [10]:
src = ColumnDataSource(distance_df)

p_cv = figure(x_axis_type='log', height=350, width=500, title='Distance coefficient of variation')
p_cv.line('dimension', 'distance_cv', source=src, line_width=2)
p_cv.scatter('dimension', 'distance_cv', source=src, size=7)
p_cv.xaxis.axis_label = 'Dimension (log scale)'
p_cv.yaxis.axis_label = 'SD(distance) / Mean(distance)'

p_rc = figure(x_axis_type='log', height=350, width=500, title='Relative distance contrast')
p_rc.line('dimension', 'relative_contrast', source=src, line_width=2)
p_rc.scatter('dimension', 'relative_contrast', source=src, size=7)
p_rc.xaxis.axis_label = 'Dimension (log scale)'
p_rc.yaxis.axis_label = '(max - min) / min'

show(row(p_cv, p_rc))

# Part II — PCA and low-rank structure

## 9. Principal Component Analysis

PCA finds orthogonal directions of maximum predictor variance.

The first component solves

\[
v_1=\arg\max_{\|v\|_2=1}v^\top\Sigma v.
\]

For centered data, SVD

\[
X=U\Sigma V^\top
\]

gives the principal directions.

### Key high-dimensional fact

Although the ambient feature space contains more than 22k coordinates, 64 centered samples can have at most 63 non-zero PCs.

So PCA separates:

- ambient dimension \(p\),
- effective sample-supported rank.

In [11]:
scaler_all = StandardScaler()
Xz = scaler_all.fit_transform(X)

max_components = min(n - 1, p)
pca_full = PCA(n_components=max_components, svd_solver='full')
Xpca = pca_full.fit_transform(Xz)

evr = pca_full.explained_variance_ratio_
cum_evr = np.cumsum(evr)

pca_summary = pd.DataFrame({
    'PC': np.arange(1, len(evr) + 1),
    'explained_variance_ratio': evr,
    'cumulative_explained_variance': cum_evr,
})

display(pca_summary.head(12))

,PC,explained_variance_ratio,cumulative_explained_variance
0,1,0.194730,0.194730
1,2,0.147440,0.342170
2,3,0.068787,0.410957
3,4,0.062265,0.473222
4,5,0.049098,0.522320
5,6,0.034201,0.556521
6,7,0.029219,0.585740
7,8,0.026138,0.611878
8,9,0.023974,0.635852
9,10,0.021239,0.657091


In [12]:
src = ColumnDataSource(pca_summary)
p_scree = figure(
    height=390,
    width=900,
    title='PCA scree plot and cumulative explained variance',
    x_axis_label='Principal component',
    y_axis_label='Variance ratio',
)
p_scree.line('PC', 'explained_variance_ratio', source=src, legend_label='Individual EVR', line_width=2)
p_scree.line('PC', 'cumulative_explained_variance', source=src, legend_label='Cumulative EVR', line_width=2)
p_scree.legend.location = 'center_right'
show(p_scree)

for threshold in [0.80, 0.90, 0.95]:
    k = int(np.searchsorted(cum_evr, threshold) + 1)
    print(f'Components for {threshold:.0%} variance: {k}')

Components for 80% variance: 22
Components for 90% variance: 36
Components for 95% variance: 47


## 10. PCA visualization

This plot is **exploratory**. We fit PCA to the entire dataset because we are not reporting its coordinates as an unbiased predictive score.

Later, PCA appears inside a `Pipeline`, so each CV fold fits its own scaler and PCA using only the training portion.

In [13]:
pca_plot_df = pd.DataFrame({
    'PC1': Xpca[:, 0],
    'PC2': Xpca[:, 1],
    'class': y.to_numpy(),
    'sample': (
        df[CFG.sample_col].astype(str).to_numpy()
        if CFG.sample_col in df.columns
        else np.arange(n).astype(str)
    ),
})

source = ColumnDataSource(pca_plot_df)
palette = Category10[max(3, len(classes))][:len(classes)]

p_pca = figure(
    height=500,
    width=900,
    title='Leukemia samples projected onto the first two PCs',
    x_axis_label=f'PC1 ({evr[0]:.1%} variance)',
    y_axis_label=f'PC2 ({evr[1]:.1%} variance)',
    tools='pan,wheel_zoom,box_zoom,reset,save',
)
p_pca.scatter(
    'PC1', 'PC2',
    source=source,
    size=10,
    alpha=0.8,
    color=factor_cmap('class', palette=palette, factors=classes),
    legend_field='class',
)
p_pca.add_tools(HoverTool(tooltips=[
    ('sample', '@sample'),
    ('class', '@class'),
    ('PC1', '@PC1{0.00}'),
    ('PC2', '@PC2{0.00}'),
]))
p_pca.legend.location = 'top_left'
show(p_pca)

## 11. PCA loadings

For component \(j\),

\[
PC_j=Xv_j.
\]

The coordinates of \(v_j\) are the **loadings**.

Large absolute loading means a gene strongly contributes to that variance direction.

But remember:

\[
\text{large variance contribution}\neq\text{large predictive importance}.
\]

In [14]:
loadings_pc1 = pd.Series(pca_full.components_[0], index=feature_cols, name='PC1_loading')

top_pc1 = (
    loadings_pc1.abs()
    .sort_values(ascending=False)
    .head(20)
    .to_frame('abs_loading')
    .join(loadings_pc1)
)

display(top_pc1)

,abs_loading,PC1_loading
221570_s_at,0.013610,0.013610
33323_r_at,0.013604,-0.013604
217722_s_at,0.013590,0.013590
203138_at,0.013577,0.013577
203721_s_at,0.013447,0.013447
212610_at,0.013395,0.013395
204274_at,0.013335,0.013335
220413_at,0.013279,-0.013279
201515_s_at,0.013240,0.013240
201486_at,0.013191,0.013191


# Part III — Sliced Inverse Regression (SIR)

## 12. Why SIR?

PCA studies the marginal distribution of \(X\), ignoring the response.

SIR studies the inverse regression structure

\[
E[X\mid Y].
\]

For categorical \(Y\), the classes naturally define slices.

A between-slice matrix is

\[
M=\sum_h p_h(\mu_h-\mu)(\mu_h-\mu)^\top.
\]

The population SIR directions arise from a generalized eigenproblem involving \(M\) and \(\Sigma_X\).

### High-dimensional complication

When \(p>n\), the empirical covariance is singular.

For this tutorial we:

1. retain a few hundred genes with marginal screening,
2. standardize,
3. add ridge regularization to the covariance,
4. solve a regularized generalized eigenproblem.

This is a pedagogical implementation, not a production SIR package.

In [15]:
class RegularizedSIR(BaseEstimator, TransformerMixin):
    """Pedagogical regularized SIR transformer for categorical y."""

    def __init__(self, n_directions: int = 2, ridge: float = 1e-2):
        self.n_directions = n_directions
        self.ridge = ridge

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)

        self.mean_ = X.mean(axis=0)
        Xc = X - self.mean_
        n_local, p_local = Xc.shape

        Sigma = (Xc.T @ Xc) / max(n_local - 1, 1)
        scale = np.trace(Sigma) / max(p_local, 1)
        Sigma_reg = Sigma + self.ridge * max(scale, 1e-12) * np.eye(p_local)

        M = np.zeros((p_local, p_local), dtype=float)
        for cls in np.unique(y):
            mask = (y == cls)
            ph = mask.mean()
            mh = Xc[mask].mean(axis=0)
            M += ph * np.outer(mh, mh)

        svals, svecs = np.linalg.eigh(Sigma_reg)
        svals = np.clip(svals, 1e-12, None)
        inv_sqrt = (svecs * (1.0 / np.sqrt(svals))) @ svecs.T

        A = inv_sqrt @ M @ inv_sqrt
        evals, evecs = np.linalg.eigh(A)
        order = np.argsort(evals)[::-1]
        k = min(self.n_directions, len(order))

        self.directions_ = inv_sqrt @ evecs[:, order[:k]]
        self.eigenvalues_ = evals[order[:k]]
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return (X - self.mean_) @ self.directions_

In [16]:
k_sir = min(300, p)
sir_screen = SelectKBest(score_func=f_classif, k=k_sir)
X_screen_sir = sir_screen.fit_transform(X, y)
X_screen_sir_z = StandardScaler().fit_transform(X_screen_sir)

sir = RegularizedSIR(n_directions=min(4, y.nunique() - 1), ridge=1e-2)
Xsir = sir.fit_transform(X_screen_sir_z, y)

sir_df = pd.DataFrame({
    'SIR1': Xsir[:, 0],
    'SIR2': Xsir[:, 1],
    'class': y.to_numpy(),
    'sample': (
        df[CFG.sample_col].astype(str).to_numpy()
        if CFG.sample_col in df.columns
        else np.arange(n).astype(str)
    ),
})

display(pd.Series(sir.eigenvalues_, name='SIR generalized eigenvalues'))

0    0.984028
1    0.983393
2    0.982630
3    0.976634
Name: SIR generalized eigenvalues, dtype: float64

In [17]:
source = ColumnDataSource(sir_df)
p_sir = figure(
    height=500,
    width=900,
    title='Regularized SIR projection (exploratory and supervised)',
    x_axis_label='SIR direction 1',
    y_axis_label='SIR direction 2',
    tools='pan,wheel_zoom,box_zoom,reset,save',
)
p_sir.scatter(
    'SIR1', 'SIR2',
    source=source,
    size=10,
    alpha=0.8,
    color=factor_cmap('class', palette=palette, factors=classes),
    legend_field='class',
)
p_sir.add_tools(HoverTool(tooltips=[('sample', '@sample'), ('class', '@class')]))
p_sir.legend.location = 'top_left'
show(p_sir)

### PCA vs SIR

| Property | PCA | SIR |
|---|---|---|
| Uses labels? | No | Yes |
| Main objective | Preserve predictor variance | Find response-relevant directions |
| Supervised? | No | Yes |
| Can be fit globally before predictive CV? | No | No |
| Main high-dimensional issue | Efficient SVD | Singular covariance/generalized eigenproblem |

A powerful conceptual distinction is:

\[
\boxed{\text{high predictor variance}\neq\text{high predictive relevance}}.
\]

# Part III-B — Subset selection and combinatorial complexity

## 13. Best subset selection

The most direct sparse model would solve

\[
\min_\beta \; \text{loss}(\beta)
\quad\text{subject to}\quad
\|\beta\|_0\le k,
\]

where \(\|\beta\|_0\) counts non-zero coefficients.

But there are

\[
2^p
\]

possible subsets of \(p\) features.

For \(p\approx22{,}283\), exhaustive search is beyond practical computation.

This is why high-dimensional analysis uses approximations such as:

- forward/backward stepwise search,
- screening + stepwise search,
- convex relaxations such as LASSO,
- mixed-integer optimization for much smaller problems.

A useful contrast is:

\[
\boxed{L_0\text{ sparsity is conceptually direct but computationally combinatorial.}}
\]

In [18]:
print(f'Number of possible subsets = 2^{p:,}')
print(f'log10(number of subsets) ≈ {p * np.log10(2):,.0f}')

Number of possible subsets = 2^22,283
log10(number of subsets) ≈ 6,708


## 14. Forward selection on a screened candidate set

Running stepwise selection over all 22k genes would still be expensive and statistically unstable.

For illustration we first keep the top 30 marginal genes and then run **forward sequential feature selection** to choose 5.

This is exploratory. If you want to report predictive performance from stepwise selection, the entire screening + stepwise process must sit inside an outer validation loop.

In [19]:
from sklearn.feature_selection import SequentialFeatureSelector

k_candidates = min(30, p)
pre_selector = SelectKBest(f_classif, k=k_candidates)
X_30 = pre_selector.fit_transform(X, y)
genes_30 = np.array(feature_cols)[pre_selector.get_support()]
X_30z = StandardScaler().fit_transform(X_30)

base_step_model = LogisticRegression(
    penalty='l2',
    C=1.0,
    solver='lbfgs',
    max_iter=5000,
    random_state=RANDOM_STATE,
)

sfs = SequentialFeatureSelector(
    base_step_model,
    n_features_to_select=5,
    direction='forward',
    scoring='f1_macro',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    n_jobs=-1,
)
sfs.fit(X_30z, y)

forward_selected_genes = genes_30[sfs.get_support()]
print('Forward-selected genes:')
print(forward_selected_genes.tolist())

Forward-selected genes:
['203434_s_at', '203435_s_at', '204749_at', '205950_s_at', '218739_at']


### Greedy complexity intuition

Forward selection does not evaluate all \(2^p\) subsets.

If it selects \(k\) variables, it evaluates roughly

\[
p+(p-1)+\cdots+(p-k+1)
\]

candidate additions, which is approximately

\[
\Theta(kp)
\]

model comparisons when \(k\ll p\).

This is far cheaper than \(\Theta(2^p)\), but it is greedy: once a variable is selected, standard forward selection does not reconsider every earlier choice.

# Part IV — SIS-style marginal screening

## 13. Cheap screening before expensive modelling

Sure Independence Screening ranks predictors using a marginal association with the response and retains only a smaller candidate set.

For a continuous response, a simple score is often

\[
|corr(X_j,Y)|.
\]

Here \(Y\) has five classes, so we use the multiclass ANOVA \(F\)-score as an analogous marginal screening statistic.

### Leakage rule

Feature screening learns from \(Y\). Therefore it belongs **inside** the CV pipeline.

Wrong:

```text
screen all rows -> CV
```

Correct:

```text
for each fold:
    fit screening on training fold
    transform train and validation folds
    fit model
    score untouched validation fold
```

In [20]:
# Exploratory ranking only. These global scores are NOT used to claim unbiased prediction.
F_scores, pvals_raw = f_classif(X, y)

screen_table = (
    pd.DataFrame({'gene': feature_cols, 'F_score': F_scores, 'raw_p': pvals_raw})
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .sort_values('F_score', ascending=False)
)

display(screen_table.head(20))

,gene,F_score,raw_p
6533,207008_at,937.392461,1.217309e-52
10604,211163_s_at,775.191998,2.997028e-50
9602,210119_at,681.426223,1.242957e-48
6047,206522_at,540.883617,9.574293e-46
3533,204007_at,450.682393,1.771635e-43
20707,221345_at,347.841137,2.776033e-40
12244,212860_at,339.784927,5.381284e-40
10232,210772_at,302.836346,1.376511e-38
19550,220187_at,302.524376,1.416939e-38
5476,205950_s_at,295.721559,2.682820e-38


In [21]:
top_plot = screen_table.head(25).sort_values('F_score')
source = ColumnDataSource(top_plot)

p_screen = figure(
    y_range=top_plot['gene'].tolist(),
    height=650,
    width=950,
    title='Top 25 genes by marginal multiclass ANOVA F-score',
    x_axis_label='F-score',
    toolbar_location=None,
)
p_screen.hbar(y='gene', right='F_score', height=0.7, source=source)
show(p_screen)

# Part V — Multiple testing and FDR

## 14. Why thousands of ordinary p-values are dangerous

If all 22,283 null hypotheses were true and each were tested at \(\alpha=0.05\), the expected number of false positives would be about

\[
22{,}283\times0.05\approx1{,}114.
\]

That is why high-dimensional inference needs multiplicity control.

In [22]:
print(f'Expected false positives at uncorrected alpha=.05 if all nulls were true: {p*0.05:,.1f}')

Expected false positives at uncorrected alpha=.05 if all nulls were true: 1,114.2


## 15. Test every gene across the five classes

For each gene \(j\), test

\[
H_{0j}:\mu_{1j}=\mu_{2j}=\cdots=\mu_{5j}.
\]

Then compare:

- raw \(p<0.05\),
- Bonferroni family-wise error control,
- Benjamini–Hochberg FDR control.

In [23]:
groups = [X.loc[y == cls].to_numpy() for cls in sorted(y.unique())]
anova_F, anova_p = stats.f_oneway(*groups, axis=0)

anova_F = np.asarray(anova_F, dtype=float)
anova_p = np.asarray(anova_p, dtype=float)
anova_F = np.where(np.isfinite(anova_F), anova_F, 0.0)
anova_p = np.where(np.isfinite(anova_p), anova_p, 1.0)

reject_bonf, p_bonf, _, _ = multipletests(anova_p, alpha=0.05, method='bonferroni')
reject_bh, p_bh, _, _ = multipletests(anova_p, alpha=0.05, method='fdr_bh')

testing_df = pd.DataFrame({
    'gene': feature_cols,
    'F': anova_F,
    'raw_p': anova_p,
    'bonferroni_p': p_bonf,
    'BH_q': p_bh,
    'reject_bonferroni': reject_bonf,
    'reject_BH_FDR_5pct': reject_bh,
})

print('Raw p < .05:', int((anova_p < .05).sum()))
print('Bonferroni discoveries:', int(reject_bonf.sum()))
print('BH-FDR discoveries:', int(reject_bh.sum()))
display(testing_df.sort_values('BH_q').head(20))

Raw p < .05: 14479
Bonferroni discoveries: 5692
BH-FDR discoveries: 13484


,gene,F,raw_p,bonferroni_p,BH_q,reject_bonferroni,reject_BH_FDR_5pct
6533,207008_at,937.392461,1.217309e-52,2.712530e-48,2.712530e-48,True,True
10604,211163_s_at,775.191998,2.997028e-50,6.678278e-46,3.339139e-46,True,True
9602,210119_at,681.426223,1.242957e-48,2.769682e-44,9.232273e-45,True,True
6047,206522_at,540.883617,9.574293e-46,2.133440e-41,5.333599e-42,True,True
3533,204007_at,450.682393,1.771635e-43,3.947735e-39,7.895470e-40,True,True
20707,221345_at,347.841137,2.776033e-40,6.185835e-36,1.030972e-36,True,True
12244,212860_at,339.784927,5.381284e-40,1.199112e-35,1.713017e-36,True,True
19550,220187_at,302.524376,1.416939e-38,3.157365e-34,3.508183e-35,True,True
10232,210772_at,302.836346,1.376511e-38,3.067280e-34,3.508183e-35,True,True
5476,205950_s_at,295.721559,2.682820e-38,5.978127e-34,5.978127e-35,True,True


## 16. Effect size vs significance

A very small p-value is not the same thing as a large scientific effect.

For one-way group comparisons, a convenient effect size is

\[
\eta^2=\frac{SS_{between}}{SS_{total}}.
\]

We plot effect size against FDR-adjusted significance.

In [24]:
X_np = X.to_numpy(dtype=float)
grand_mean = X_np.mean(axis=0)
ss_total = ((X_np - grand_mean) ** 2).sum(axis=0)
ss_between = np.zeros(p)

for cls in y.unique():
    Xc = X.loc[y == cls].to_numpy(dtype=float)
    class_mean = Xc.mean(axis=0)
    ss_between += Xc.shape[0] * (class_mean - grand_mean) ** 2

eta2 = np.divide(ss_between, ss_total, out=np.zeros_like(ss_between), where=ss_total > 0)

testing_df['eta_squared'] = eta2
testing_df['minus_log10_q'] = -np.log10(np.clip(testing_df['BH_q'], 1e-300, 1.0))

top_fdr = testing_df.nsmallest(500, 'BH_q').copy()
source = ColumnDataSource(top_fdr)

p_effect = figure(
    height=500,
    width=900,
    title='Top 500 FDR-ranked genes: effect size vs adjusted significance',
    x_axis_label='Eta-squared effect size',
    y_axis_label='-log10(BH q-value)',
    tools='pan,wheel_zoom,box_zoom,reset,save',
)
p_effect.scatter('eta_squared', 'minus_log10_q', source=source, size=7, alpha=0.65)
p_effect.add_tools(HoverTool(tooltips=[
    ('gene', '@gene'),
    ('eta²', '@eta_squared{0.000}'),
    ('BH q', '@BH_q{0.00e}'),
]))
show(p_effect)

### Bonferroni vs BH-FDR

Bonferroni controls the family-wise error rate:

\[
FWER=P(V\ge1).
\]

It is intentionally strict.

Benjamini–Hochberg controls

\[
FDR=E\left[\frac{V}{R\vee1}\right].
\]

With sorted p-values \(p_{(1)}\le\cdots\le p_{(m)}\), BH finds the largest \(k\) satisfying

\[
p_{(k)}\le\frac{k}{m}q.
\]

For large-scale exploratory discovery, FDR often offers a more useful trade-off than controlling the probability of even one false positive.

# Part VI — Ridge, LASSO, and Elastic Net

## 17. Ridge

For least squares,

\[
\hat\beta^{ridge}=
\arg\min_\beta\left[\|y-X\beta\|_2^2+\lambda\|\beta\|_2^2\right]
\]

with solution

\[
\hat\beta^{ridge}=(X^\top X+\lambda I)^{-1}X^\top y.
\]

Ridge stabilizes poorly supported eigen-directions and usually keeps all coefficients non-zero.

## 18. LASSO

\[
\hat\beta^{lasso}=
\arg\min_\beta\left[\text{loss}(\beta)+\lambda\|\beta\|_1\right].
\]

The \(L_1\) geometry produces exact zeros, yielding a sparse model.

## 19. Elastic Net

\[
\lambda\left[\alpha\|\beta\|_1+(1-\alpha)\frac{\|\beta\|_2^2}{2}\right].
\]

This can be more stable than pure LASSO when predictors are highly correlated.

## 20. Leakage-safe predictive pipelines

We compare several structural assumptions:

- `Dummy`: no useful signal.
- `PCA + Ridge Logistic`: low-rank representation.
- `SIS + Ridge Logistic`: screened dense linear model.
- `SIS + LASSO Logistic`: screened sparse linear model.
- `SIS + Elastic Net`: sparse model with extra stability.
- `PCA + kNN`: local geometry after dimension reduction.
- `SIS + Decision Tree`: nonlinear partitioning.
- `SIS + Random Forest`: bagged/decorrelated trees.
- `SIS + MLP`: small regularized nonlinear model.

The screening, scaling, and PCA steps are all *inside* pipelines.

In [25]:
screen_k = min(CFG.screening_k, p)

models: Dict[str, BaseEstimator] = {
    'Dummy': DummyClassifier(strategy='most_frequent'),

    'PCA + Ridge Logistic': Pipeline([
        ('scale', StandardScaler()),
        ('pca', PCA(n_components=0.90, svd_solver='full')),
        ('clf', LogisticRegression(
            penalty='l2', C=1.0, solver='lbfgs', max_iter=5000,
            random_state=RANDOM_STATE,
        )),
    ]),

    'SIS + Ridge Logistic': Pipeline([
        ('screen', SelectKBest(f_classif, k=screen_k)),
        ('scale', StandardScaler()),
        ('clf', LogisticRegression(
            penalty='l2', C=1.0, solver='lbfgs', max_iter=5000,
            random_state=RANDOM_STATE,
        )),
    ]),

    'SIS + LASSO Logistic': Pipeline([
        ('screen', SelectKBest(f_classif, k=screen_k)),
        ('scale', StandardScaler()),
        ('clf', LogisticRegression(
            penalty='l1', C=0.2, solver='saga', max_iter=10000,
            random_state=RANDOM_STATE,
        )),
    ]),

    'SIS + Elastic Net Logistic': Pipeline([
        ('screen', SelectKBest(f_classif, k=screen_k)),
        ('scale', StandardScaler()),
        ('clf', LogisticRegression(
            penalty='elasticnet', l1_ratio=0.5, C=0.2,
            solver='saga', max_iter=10000, random_state=RANDOM_STATE,
        )),
    ]),

    'PCA + kNN': Pipeline([
        ('scale', StandardScaler()),
        ('pca', PCA(n_components=0.90, svd_solver='full')),
        ('clf', KNeighborsClassifier(n_neighbors=5)),
    ]),

    'SIS + Decision Tree': Pipeline([
        ('screen', SelectKBest(f_classif, k=screen_k)),
        ('clf', DecisionTreeClassifier(
            max_depth=4, min_samples_leaf=3, random_state=RANDOM_STATE,
        )),
    ]),

    'SIS + Random Forest': Pipeline([
        ('screen', SelectKBest(f_classif, k=min(500, p))),
        ('clf', RandomForestClassifier(
            n_estimators=250 if CFG.fast_mode else 600,
            max_features='sqrt', min_samples_leaf=2,
            n_jobs=1, random_state=RANDOM_STATE,
        )),
    ]),

    'SIS + MLP': Pipeline([
        ('screen', SelectKBest(f_classif, k=screen_k)),
        ('scale', StandardScaler()),
        ('clf', MLPClassifier(
            hidden_layer_sizes=(32,), alpha=1e-2,
            early_stopping=True, validation_fraction=0.20,
            max_iter=1500, random_state=RANDOM_STATE,
        )),
    ]),
}

list(models)

['Dummy',
 'PCA + Ridge Logistic',
 'SIS + Ridge Logistic',
 'SIS + LASSO Logistic',
 'SIS + Elastic Net Logistic',
 'PCA + kNN',
 'SIS + Decision Tree',
 'SIS + Random Forest',
 'SIS + MLP']

## 21. Repeated stratified cross-validation

A single train/test split is too unstable for 64 observations.

We use repeated stratified four-fold CV and report:

\[
Accuracy,
\quad Balanced\ Accuracy,
\quad F1_{macro}.
\]

The spread across folds is as important as the mean.

In [26]:
from dataclasses import dataclass
from typing import Dict

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator
from sklearn.model_selection import cross_validate, RepeatedStratifiedKFold
from sklearn.preprocessing import LabelEncoder


@dataclass
class ExperimentRunner:
    models: Dict[str, BaseEstimator]
    cv: object

    def evaluate(
        self,
        X: pd.DataFrame,
        y: pd.Series
    ) -> pd.DataFrame:

        # ---------------------------------------------------------
        # 1. Force X to a pure numeric matrix
        # ---------------------------------------------------------
        X_numeric = (
            X.apply(pd.to_numeric, errors="coerce")
            .astype(np.float64)
        )

        # Handle NaN values defensively
        if X_numeric.isna().any().any():
            print("NaNs detected in X -> applying median imputation")

            X_numeric = X_numeric.fillna(
                X_numeric.median()
            )

        X_np = X_numeric.to_numpy(
            dtype=np.float64,
            copy=False
        )

        # Check infinity
        if not np.isfinite(X_np).all():
            raise ValueError(
                "X contains +inf or -inf values."
            )

        # ---------------------------------------------------------
        # 2. Encode string labels
        #
        # AML, ALL, CLL, CML, ...
        #       ↓
        # 0,   1,   2,   3, ...
        # ---------------------------------------------------------
        label_encoder = LabelEncoder()

        y_np = label_encoder.fit_transform(
            y.astype(str).to_numpy()
        ).astype(np.int64)

        print(
            "Class mapping:",
            dict(
                zip(
                    label_encoder.classes_,
                    range(len(label_encoder.classes_))
                )
            )
        )

        print(
            f"X dtype: {X_np.dtype}, "
            f"shape: {X_np.shape}"
        )

        print(
            f"y dtype: {y_np.dtype}, "
            f"shape: {y_np.shape}"
        )

        # ---------------------------------------------------------
        # 3. Scoring
        # ---------------------------------------------------------
        scoring = {
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy",
            "macro_f1": "f1_macro",
        }

        rows = []

        # ---------------------------------------------------------
        # 4. Evaluate models
        # ---------------------------------------------------------
        for name, model in self.models.items():

            print(f"\nEvaluating: {name}")

            try:
                scores = cross_validate(
                    estimator=model,
                    X=X_np,
                    y=y_np,
                    cv=self.cv,
                    scoring=scoring,
                    n_jobs=-1,
                    return_train_score=False,
                    error_score="raise",
                )

                rows.append({
                    "model": name,

                    "accuracy_mean":
                        np.mean(scores["test_accuracy"]),

                    "accuracy_sd":
                        np.std(
                            scores["test_accuracy"],
                            ddof=1
                        ),

                    "balanced_accuracy_mean":
                        np.mean(
                            scores[
                                "test_balanced_accuracy"
                            ]
                        ),

                    "balanced_accuracy_sd":
                        np.std(
                            scores[
                                "test_balanced_accuracy"
                            ],
                            ddof=1
                        ),

                    "macro_f1_mean":
                        np.mean(
                            scores["test_macro_f1"]
                        ),

                    "macro_f1_sd":
                        np.std(
                            scores["test_macro_f1"],
                            ddof=1
                        ),
                })

            except Exception as exc:

                print(
                    f"FAILED: {name}\n"
                    f"{type(exc).__name__}: {exc}"
                )

                raise

        # ---------------------------------------------------------
        # 5. Results
        # ---------------------------------------------------------
        result_df = pd.DataFrame(rows)

        return (
            result_df
            .sort_values(
                [
                    "macro_f1_mean",
                    "balanced_accuracy_mean"
                ],
                ascending=False
            )
            .reset_index(drop=True)
        )


# =============================================================
# Cross-validation
# =============================================================

cv = RepeatedStratifiedKFold(
    n_splits=4,
    n_repeats=2 if CFG.fast_mode else 8,
    random_state=RANDOM_STATE,
)

runner = ExperimentRunner(
    models=models,
    cv=cv
)

benchmark_df = runner.evaluate(
    X=X,
    y=y
)

display(benchmark_df)

Class mapping: {'AML': 0, 'Bone_Marrow': 1, 'Bone_Marrow_CD34': 2, 'PB': 3, 'PBSC_CD34': 4}
X dtype: float64, shape: (64, 22283)
y dtype: int64, shape: (64,)

Evaluating: Dummy

Evaluating: PCA + Ridge Logistic

Evaluating: SIS + Ridge Logistic

Evaluating: SIS + LASSO Logistic

Evaluating: SIS + Elastic Net Logistic

Evaluating: PCA + kNN

Evaluating: SIS + Decision Tree

Evaluating: SIS + Random Forest

Evaluating: SIS + MLP


,model,accuracy_mean,accuracy_sd,balanced_accuracy_mean,balanced_accuracy_sd,macro_f1_mean,macro_f1_sd
0,SIS + Ridge Logistic,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000
1,SIS + Elastic Net Logistic,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000
2,SIS + Random Forest,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000
3,SIS + LASSO Logistic,0.992188,0.022097,0.987500,0.035355,0.986667,0.037712
4,PCA + Ridge Logistic,0.968750,0.033408,0.984524,0.016642,0.974466,0.027822
5,SIS + MLP,0.914062,0.057258,0.892262,0.081249,0.897910,0.070799
6,SIS + Decision Tree,0.906250,0.074702,0.896429,0.077737,0.879632,0.097887
7,PCA + kNN,0.804688,0.052158,0.887500,0.039998,0.824639,0.045032
8,Dummy,0.406250,0.033408,0.200000,0.000000,0.115415,0.006761


In [27]:
plot_df = benchmark_df.sort_values('macro_f1_mean').copy()
source = ColumnDataSource(plot_df)

p_models = figure(
    y_range=plot_df['model'].tolist(),
    height=520,
    width=980,
    title='Repeated-CV model comparison',
    x_axis_label='Mean macro-F1',
    x_range=(0, 1.05),
    toolbar_location=None,
)
p_models.hbar(y='model', right='macro_f1_mean', height=0.7, source=source)
show(p_models)

### How to read the benchmark

Do not treat the first row as “the truth.”

With \(n=64\), the standard deviation across resamples is itself an important finding.

The more useful questions are structural:

- Does PCA preserve enough discriminative signal?
- Does screening reduce variance?
- Does LASSO become too sparse?
- Do tree ensembles gain enough nonlinear flexibility to justify added variance?
- Does k-NN remain sensitive to high-dimensional geometry?
- Does the MLP overfit despite regularization?

High-dimensional analysis should emphasize **stability and uncertainty**, not just a leaderboard.

# Part VII — LASSO sparsity path

## 22. Regularization strength and active variables

scikit-learn uses \(C\), approximately inverse regularization strength:

\[
C\uparrow\Rightarrow\lambda\downarrow.
\]

Therefore:

- small \(C\): strong shrinkage, fewer active genes,
- large \(C\): weak shrinkage, more active genes.

For multiclass logistic regression, a feature is called active if at least one class coefficient is non-zero.

In [28]:
# Exploratory full-data path; do not interpret as unbiased biomarker discovery.
selector_path = SelectKBest(f_classif, k=screen_k)
X_sel = selector_path.fit_transform(X, y)
selected_genes = np.array(feature_cols)[selector_path.get_support()]
X_sel_z = StandardScaler().fit_transform(X_sel)

C_grid = np.logspace(-3, 1.5, 14)
path_rows = []

for C in C_grid:
    clf = LogisticRegression(
        penalty='l1', C=float(C), solver='saga',
        max_iter=15000, random_state=RANDOM_STATE,
    )
    clf.fit(X_sel_z, y)
    active = np.any(np.abs(clf.coef_) > 1e-8, axis=0)
    path_rows.append({
        'C': C,
        'active_features': int(active.sum()),
        'coef_l1_norm': float(np.abs(clf.coef_).sum()),
    })

lasso_path_df = pd.DataFrame(path_rows)
display(lasso_path_df)

,C,active_features,coef_l1_norm
0,0.001000,0,0.000000
1,0.002219,0,0.000000
2,0.004924,0,0.000000
3,0.010926,0,0.000000
4,0.024245,0,0.000000
5,0.053798,7,1.014574
6,0.119378,13,4.536708
7,0.264897,20,7.322255
8,0.587802,29,9.947280
9,1.304321,44,12.753706


In [29]:
source = ColumnDataSource(lasso_path_df)
p_lasso_path = figure(
    x_axis_type='log',
    height=400,
    width=880,
    title='LASSO logistic sparsity path',
    x_axis_label='C (larger = weaker regularization)',
    y_axis_label='Number of active screened genes',
)
p_lasso_path.line('C', 'active_features', source=source, line_width=2)
p_lasso_path.scatter('C', 'active_features', source=source, size=7)
show(p_lasso_path)

## 23. Inspect one sparse model

This is useful for understanding coefficient sparsity, but it is **not** a validated biomarker list because the same full dataset was used for screening and coefficient estimation.

In [30]:
sparse_model = LogisticRegression(
    penalty='l1', C=0.2, solver='saga',
    max_iter=15000, random_state=RANDOM_STATE,
)
sparse_model.fit(X_sel_z, y)

coef_abs_max = np.max(np.abs(sparse_model.coef_), axis=0)
coef_df = (
    pd.DataFrame({
        'gene': selected_genes,
        'max_abs_coefficient': coef_abs_max,
        'active': coef_abs_max > 1e-8,
    })
    .sort_values('max_abs_coefficient', ascending=False)
)

print('Active screened genes:', int(coef_df['active'].sum()))
display(coef_df.head(30))

Active screened genes: 16


,gene,max_abs_coefficient,active
35,204122_at,0.798129,True
109,210976_s_at,0.794132,True
191,221477_s_at,0.772566,True
51,205413_at,0.717576,True
118,211820_x_at,0.712247,True
42,204754_at,0.685871,True
110,211163_s_at,0.462700,True
74,207008_at,0.393838,True
159,216510_x_at,0.244160,True
96,210119_at,0.176841,True


# Part VII-B — Bootstrap selection stability

## 24. Sparse models can predict stably while selecting genes unstably

With correlated predictors and tiny \(n\), LASSO may choose different members of a correlated gene group after small perturbations of the sample.

A useful diagnostic is **selection frequency**:

1. bootstrap observations,
2. redo screening,
3. refit the sparse model,
4. record which genes are active,
5. estimate

\[
\hat\pi_j=\frac{1}{B}\sum_{b=1}^B I(j\text{ selected in bootstrap }b).
\]

This is a practical stability diagnostic. It is not automatically a formal confidence probability.

The code is optional because it repeats many model fits.

In [31]:
RUN_BOOTSTRAP_STABILITY = False

if RUN_BOOTSTRAP_STABILITY:
    B = 50 if CFG.fast_mode else 200
    boot_screen_k = min(100, p)
    selection_counts = pd.Series(0, index=feature_cols, dtype=int)

    # Stratified bootstrap: resample inside each class so every replicate retains all classes.
    class_indices = {
        cls: np.flatnonzero(y.to_numpy() == cls)
        for cls in y.unique()
    }

    for b in range(B):
        boot_idx_parts = [
            rng.choice(idx_cls, size=len(idx_cls), replace=True)
            for idx_cls in class_indices.values()
        ]
        boot_idx = np.concatenate(boot_idx_parts)
        rng.shuffle(boot_idx)

        Xb = X.iloc[boot_idx]
        yb = y.iloc[boot_idx]

        sel = SelectKBest(f_classif, k=boot_screen_k)
        Xb_sel = sel.fit_transform(Xb, yb)
        genes_b = np.array(feature_cols)[sel.get_support()]
        Xb_sel = StandardScaler().fit_transform(Xb_sel)

        clf = LogisticRegression(
            penalty='l1',
            C=0.2,
            solver='saga',
            max_iter=10000,
            random_state=RANDOM_STATE + b,
        )
        clf.fit(Xb_sel, yb)

        active = np.any(np.abs(clf.coef_) > 1e-8, axis=0)
        selection_counts.loc[genes_b[active]] += 1

    stability_df = (
        pd.DataFrame({
            'gene': selection_counts.index,
            'selection_frequency': selection_counts.values / B,
        })
        .sort_values('selection_frequency', ascending=False)
        .reset_index(drop=True)
    )

    display(stability_df.head(30))

    top_stable = stability_df.head(25).sort_values('selection_frequency')
    source = ColumnDataSource(top_stable)
    p_stability = figure(
        y_range=top_stable['gene'].tolist(),
        height=650,
        width=950,
        title='Bootstrap sparse-selection frequency',
        x_axis_label='Selection frequency',
        x_range=(0, 1.0),
        toolbar_location=None,
    )
    p_stability.hbar(y='gene', right='selection_frequency', height=0.7, source=source)
    show(p_stability)
else:
    print('Set RUN_BOOTSTRAP_STABILITY=True to estimate gene-selection frequencies.')

Set RUN_BOOTSTRAP_STABILITY=True to estimate gene-selection frequencies.


### What instability means

Suppose genes \(X_1\) and \(X_2\) are highly correlated and both carry nearly the same signal.

Two bootstrap samples may produce

\[
\hat\beta_1\neq0,\;\hat\beta_2=0
\]

and

\[
\hat\beta_1=0,\;\hat\beta_2\neq0
\]

while predictions remain nearly unchanged.

Thus:

\[
\boxed{\text{prediction stability does not imply variable-selection stability}.}
\]

This distinction is critical whenever selected features are interpreted scientifically.

# Part VIII — Nested cross-validation

## 24. Separating tuning from performance estimation

If we use CV both to tune \(k\) and \(C\) and to report the final score, model selection can leak into performance estimation.

Nested CV separates the jobs:

```text
outer CV  -> estimate generalization
    inner CV -> select k, penalty and C
```

This is especially important in high dimension because searching many preprocessing/model combinations can itself overfit.

The next cell is off by default because it runs many model fits. Toggle it on when you want the rigorous model-selection experiment.

In [32]:
RUN_NESTED_CV = False

if RUN_NESTED_CV:
    base_pipe = Pipeline([
        ('screen', SelectKBest(f_classif)),
        ('scale', StandardScaler()),
        ('clf', LogisticRegression(
            solver='saga', max_iter=15000, random_state=RANDOM_STATE,
        )),
    ])

    param_grid = [
        {
            'screen__k': [50, 100, 200, 500],
            'clf__penalty': ['l1', 'l2'],
            'clf__C': [0.03, 0.1, 0.3, 1.0, 3.0],
        },
        {
            'screen__k': [50, 100, 200, 500],
            'clf__penalty': ['elasticnet'],
            'clf__l1_ratio': [0.25, 0.5, 0.75],
            'clf__C': [0.03, 0.1, 0.3, 1.0],
        },
    ]

    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    outer_cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE + 1)

    search = GridSearchCV(
        base_pipe,
        param_grid=param_grid,
        scoring='f1_macro',
        cv=inner_cv,
        n_jobs=-1,
    )

    nested_scores = cross_validate(
        search,
        X,
        y,
        scoring={
            'accuracy': 'accuracy',
            'balanced_accuracy': 'balanced_accuracy',
            'macro_f1': 'f1_macro',
        },
        cv=outer_cv,
        n_jobs=1,
        return_estimator=True,
    )

    print('Nested macro-F1 mean:', nested_scores['test_macro_f1'].mean())
    print('Nested macro-F1 sd  :', nested_scores['test_macro_f1'].std(ddof=1))
    display(pd.DataFrame([est.best_params_ for est in nested_scores['estimator']]))
else:
    print('Set RUN_NESTED_CV=True to run nested model selection.')

Set RUN_NESTED_CV=True to run nested model selection.


# Part IX — Trees, bagging and random forests

## 25. Why trees are useful here

A tree adaptively chooses variables and split points. Only features appearing in splits affect the fitted tree, so a tree performs an implicit form of variable selection.

But a single tree has high variance.

Bagging averages trees fitted to bootstrap samples. A rough correlated-ensemble variance expression is

\[
\rho\sigma^2 + \frac{1-\rho}{B}\sigma^2,
\]

where \(B\) is the number of trees and \(\rho\) is pairwise correlation.

Random forests further reduce correlation by randomizing candidate variables at each split.

In our benchmark we still screen first to make the experiment computationally economical in a 22k-feature space.

# Part X — Unsupervised structure

## 26. K-means after PCA

Direct clustering in tens of thousands of dimensions is difficult to interpret and can be sensitive to distance concentration.

A common workflow is

\[
X\to standardize\to PCA\to K\text{-means}.
\]

After clustering, we compare clusters with known labels using permutation-invariant metrics:

- Adjusted Rand Index (ARI)
- Normalized Mutual Information (NMI)

The labels are **not** used to fit the clusters.

In [33]:
n_cluster_pcs = min(10, n - 1)
X_cluster = PCA(n_components=n_cluster_pcs).fit_transform(StandardScaler().fit_transform(X))

kmeans = KMeans(n_clusters=y.nunique(), n_init=50, random_state=RANDOM_STATE)
k_labels = kmeans.fit_predict(X_cluster)

agg = AgglomerativeClustering(n_clusters=y.nunique())
a_labels = agg.fit_predict(X_cluster)

cluster_metrics = pd.DataFrame([
    {
        'method': 'KMeans',
        'ARI': adjusted_rand_score(y, k_labels),
        'NMI': normalized_mutual_info_score(y, k_labels),
    },
    {
        'method': 'Agglomerative',
        'ARI': adjusted_rand_score(y, a_labels),
        'NMI': normalized_mutual_info_score(y, a_labels),
    },
])

display(cluster_metrics)

,method,ARI,NMI
0,KMeans,0.438195,0.656221
1,Agglomerative,0.319049,0.602690


In [34]:
cluster_plot_df = pd.DataFrame({
    'PC1': X_cluster[:, 0],
    'PC2': X_cluster[:, 1],
    'true_class': y.to_numpy(),
    'kmeans_cluster': k_labels.astype(str),
})

cluster_factors = sorted(cluster_plot_df['kmeans_cluster'].unique().tolist())
cluster_palette = Category10[max(3, len(cluster_factors))][:len(cluster_factors)]
source = ColumnDataSource(cluster_plot_df)

p_true = figure(height=430, width=510, title='PCA space: true class', x_axis_label='PC1', y_axis_label='PC2')
p_true.scatter(
    'PC1', 'PC2', source=source, size=9,
    color=factor_cmap('true_class', palette=palette, factors=classes),
)

p_km = figure(height=430, width=510, title='PCA space: K-means cluster', x_axis_label='PC1', y_axis_label='PC2')
p_km.scatter(
    'PC1', 'PC2', source=source, size=9,
    color=factor_cmap('kmeans_cluster', palette=cluster_palette, factors=cluster_factors),
)

show(row(p_true, p_km))

### Classification and clustering answer different questions

Classification uses \(Y\) and asks for a decision boundary.

Clustering ignores \(Y\) and asks whether the marginal geometry of \(X\) naturally partitions into groups.

Therefore

\[
\boxed{\text{supervised separability}\neq\text{unsupervised clusterability}}.
\]

# Part XI — Prediction vs inference

## 27. Simple sample splitting for post-selection inference

A dangerous workflow is:

1. examine all 22k genes,
2. choose the strongest 10,
3. test those same genes on the same observations.

The p-values no longer reflect a pre-specified hypothesis because selection already exploited random fluctuations.

A simple educational remedy is:

```text
discovery half -> select candidate genes
inference half -> test only those pre-chosen genes
```

This wastes data and therefore loses power, but it makes the independence principle visible.

In [35]:
X_disc, X_inf, y_disc, y_inf = train_test_split(
    X, y,
    test_size=0.50,
    stratify=y,
    random_state=RANDOM_STATE,
)

k_discovery = min(10, p)
selector_disc = SelectKBest(f_classif, k=k_discovery)
selector_disc.fit(X_disc, y_disc)
chosen_genes = np.array(feature_cols)[selector_disc.get_support()]

print('Genes selected using discovery split only:')
print(chosen_genes.tolist())

Genes selected using discovery split only:
['204749_at', '205798_at', '205950_s_at', '206522_at', '207008_at', '210119_at', '211163_s_at', '213122_at', '219672_at', '220187_at']


In [36]:
holdout_pvals = []

for gene in chosen_genes:
    samples_by_class = [
        X_inf.loc[y_inf == cls, gene].to_numpy()
        for cls in sorted(y_inf.unique())
    ]
    _, pval = stats.f_oneway(*samples_by_class)
    holdout_pvals.append(pval)

reject_holdout, adjusted_holdout, _, _ = multipletests(
    holdout_pvals,
    alpha=0.05,
    method='holm',
)

sample_split_inference = (
    pd.DataFrame({
        'gene': chosen_genes,
        'holdout_raw_p': holdout_pvals,
        'holdout_Holm_p': adjusted_holdout,
        'reject_5pct': reject_holdout,
    })
    .sort_values('holdout_Holm_p')
)

display(sample_split_inference)

,gene,holdout_raw_p,holdout_Holm_p,reject_5pct
4,207008_at,2.137030e-24,2.137030e-23,True
5,210119_at,2.433177e-24,2.189859e-23,True
6,211163_s_at,5.411908e-23,4.329527e-22,True
3,206522_at,8.244965e-20,5.771475e-19,True
9,220187_at,5.699184e-17,3.419511e-16,True
2,205950_s_at,4.855035e-16,2.427517e-15,True
0,204749_at,5.573509e-15,2.229404e-14,True
8,219672_at,7.955787e-13,2.386736e-12,True
7,213122_at,6.029863e-08,1.205973e-07,True
1,205798_at,1.441316e-07,1.441316e-07,True


Do not be surprised if few genes survive.

That is an important high-dimensional lesson:

- good prediction can exist even when individual-variable inference is weak,
- valid inference may require much more data,
- selection uncertainty is real.

\[
\boxed{\text{prediction}\neq\text{inference}}.
\]

# Part XII — Hidden Markov Models: knowing when *not* to use a method

## 28. Why an HMM does not match the Kaggle dataset

An HMM assumes an ordered sequence of latent states

\[
Z_1,Z_2,\ldots,Z_T
\]

and observations

\[
X_1,X_2,\ldots,X_T.
\]

The Markov assumption is

\[
P(Z_t\mid Z_{1:t-1})=P(Z_t\mid Z_{t-1}).
\]

The leukemia rows are separate biological samples. There is no scientifically meaningful temporal order linking row 1 to row 2.

Arbitrarily treating the rows as a sequence would create a false dependency.

**A strong data scientist should be able to reject an algorithm whose assumptions do not fit the data.**

The next section is therefore a separate synthetic HMM example, included only to connect this case-study notebook to the broader DSA4211 syllabus.

## 29. Minimal discrete HMM

For \(K\) hidden states and sequence length \(T\):

### Forward algorithm

\[
\alpha_t(j)=P(x_{1:t},Z_t=j)
\]

with recurrence

\[
\alpha_t(j)=\left[\sum_i\alpha_{t-1}(i)A_{ij}\right]B_j(x_t).
\]

### Viterbi algorithm

\[
\hat z_{1:T}=\arg\max_{z_{1:T}}P(z_{1:T},x_{1:T}).
\]

Both have time complexity

\[
\Theta(TK^2).
\]

In [37]:
class DiscreteHMM:
    def __init__(self, initial, transition, emission):
        self.pi = np.asarray(initial, dtype=float)
        self.A = np.asarray(transition, dtype=float)
        self.B = np.asarray(emission, dtype=float)

    def forward_probability(self, observations: Iterable[int]) -> float:
        obs = list(observations)
        alpha = self.pi * self.B[:, obs[0]]
        for x_t in obs[1:]:
            alpha = (alpha @ self.A) * self.B[:, x_t]
        return float(alpha.sum())

    def viterbi(self, observations: Iterable[int]) -> Tuple[np.ndarray, float]:
        obs = list(observations)
        T = len(obs)
        K = len(self.pi)

        log_pi = np.log(self.pi)
        log_A = np.log(self.A)
        log_B = np.log(self.B)

        dp = np.empty((T, K))
        back = np.zeros((T, K), dtype=int)
        dp[0] = log_pi + log_B[:, obs[0]]

        for t in range(1, T):
            for j in range(K):
                candidates = dp[t - 1] + log_A[:, j]
                back[t, j] = np.argmax(candidates)
                dp[t, j] = candidates[back[t, j]] + log_B[j, obs[t]]

        path = np.zeros(T, dtype=int)
        path[-1] = np.argmax(dp[-1])
        for t in range(T - 2, -1, -1):
            path[t] = back[t + 1, path[t + 1]]

        return path, float(np.exp(dp[-1, path[-1]]))


hmm = DiscreteHMM(
    initial=[0.6, 0.4],
    transition=[[0.85, 0.15], [0.20, 0.80]],
    emission=[[0.70, 0.20, 0.10], [0.10, 0.30, 0.60]],
)

observations = [0, 0, 1, 2, 2, 1, 0, 0]
prob = hmm.forward_probability(observations)
path, joint_prob = hmm.viterbi(observations)

print('Observations:', observations)
print('P(observations):', prob)
print('Viterbi state path:', path.tolist())
print('Joint probability of Viterbi path and observations:', joint_prob)

Observations: [0, 0, 1, 2, 2, 1, 0, 0]
P(observations): 0.00025053433017468746
Viterbi state path: [0, 0, 1, 1, 1, 1, 0, 0]
Joint probability of Viterbi path and observations: 5.17985362944e-05


# Part XIII — Computational complexity

## 30. Why numerical linear algebra matters

Approximate costs:

| Operation | Typical complexity |
|---|---:|
| Construct \(X^\top X\) | \(\Theta(np^2)\) |
| Dense \(p\times p\) inverse | \(\Theta(p^3)\) |
| Full covariance PCA | roughly \(O(np^2+p^3)\) |
| Thin/truncated SVD | much cheaper when effective rank \(k\ll p\) |
| K-means iteration | \(\Theta(nKp)\) |
| Sparse linear-model coordinate sweep | roughly \(O(np)\) |
| Exhaustive best-subset search | \(\Theta(2^p)\) subsets |
| HMM forward/Viterbi | \(\Theta(TK^2)\) |

For this dataset, a full \(p\times p\) Gram matrix contains nearly 500 million entries.

At 8 bytes per float, that is close to 4 GiB just for one dense array, before temporary arrays and Python overhead.

In [38]:
gram_entries = p * p
approx_gib = gram_entries * 8 / (1024 ** 3)
print(f'Full p x p Gram entries: {gram_entries:,}')
print(f'Float64 storage alone: ~{approx_gib:.2f} GiB')

Full p x p Gram entries: 496,532,089
Float64 storage alone: ~3.70 GiB


# Part XIV — A reusable high-dimensional workflow

## 31. Mental model

```text
1. Audit n, p, class sizes, missingness
             ↓
2. Define goal:
   prediction? inference? discovery?
             ↓
3. Define split/CV strategy before learning transforms
             ↓
4. Inside each training fold:
      scaling
      screening / PCA / supervised reduction
      regularization
      model fitting
             ↓
5. Evaluate on untouched validation data
             ↓
6. Repeat across folds/resamples
             ↓
7. Inspect uncertainty and stability
             ↓
8. For thousands of tests:
      FWER or FDR control
             ↓
9. For scientific claims:
      post-selection inference and external validation
```

The recurring DSA4211 principle is

\[
\boxed{
\text{high dimensionality + limited samples}
\Rightarrow
\text{structure + regularization + careful validation}
}.
\]

# Part XV — Exercises

### Exercise 1 — Rank
Prove that the centered \(64\times22{,}283\) design matrix has rank at most 63.

### Exercise 2 — Leakage
Fit `SelectKBest` once to the entire dataset before cross-validation. Compare this with the proper Pipeline result. Explain the optimism.

### Exercise 3 — Screening size
Using nested CV, compare

\[
k\in\{25,50,100,200,500,1000\}
\]

for sparse logistic regression.

### Exercise 4 — PCA
Find the number of PCs required to explain 80%, 90%, and 95% of standardized variance.

### Exercise 5 — Multiplicity
Compare discoveries under:

- raw \(p<0.05\),
- Bonferroni,
- Holm,
- BH-FDR 1%,
- BH-FDR 5%,
- BH-FDR 10%.

### Exercise 6 — Stability selection
Bootstrap the dataset 100 times. Refit a sparse model each time and estimate each gene's selection frequency.

### Exercise 7 — Clustering
Compare K-means ARI after 2, 5, 10, 20, and 40 PCs.

### Exercise 8 — Prediction vs inference
Find two models with similar macro-F1 but different feature-selection stability. Explain why predictive equivalence does not imply inferential equivalence.

# Part XVI — Worked solutions

## Solution 1 — Rank

For any matrix,

\[
rank(X)\le\min(n,p).
\]

For the centered design \(X_c\), the centered rows satisfy

\[
\mathbf{1}^\top X_c=0.
\]

Thus the 64 rows are linearly dependent and

\[
rank(X_c)\le63.
\]

In [39]:
X_centered = X.to_numpy() - X.to_numpy().mean(axis=0, keepdims=True)
row_sum_norm = np.linalg.norm(X_centered.sum(axis=0))
print('||sum of centered rows||_2 =', row_sum_norm)
print('Theoretical centered rank upper bound =', n - 1)

||sum of centered rows||_2 = 4.845728796406784e-12
Theoretical centered rank upper bound = 63


## Solution 2 — Deliberately demonstrate leakage

The following code is **wrong on purpose**.

It uses all labels to choose genes before CV, allowing information from validation rows to influence which variables enter the model.

In [40]:
# DELIBERATELY LEAKY: educational anti-pattern
leaky_selector = SelectKBest(f_classif, k=screen_k)
X_leaky = leaky_selector.fit_transform(X, y)

leaky_model = Pipeline([
    ('scale', StandardScaler()),
    ('clf', LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
])

leaky_scores = cross_validate(
    leaky_model, X_leaky, y,
    cv=cv,
    scoring={'macro_f1': 'f1_macro'},
    n_jobs=-1,
)

proper_scores = cross_validate(
    models['SIS + Ridge Logistic'], X, y,
    cv=cv,
    scoring={'macro_f1': 'f1_macro'},
    n_jobs=-1,
)

print('LEAKY mean macro-F1 :', leaky_scores['test_macro_f1'].mean())
print('PROPER mean macro-F1:', proper_scores['test_macro_f1'].mean())

LEAKY mean macro-F1 : 1.0
PROPER mean macro-F1: 1.0


## Solution 4 — PCA thresholds

In [41]:
threshold_results = []
for threshold in [0.80, 0.90, 0.95]:
    threshold_results.append({
        'target_explained_variance': threshold,
        'n_components': int(np.searchsorted(cum_evr, threshold) + 1),
    })
display(pd.DataFrame(threshold_results))

,target_explained_variance,n_components
0,0.80,22
1,0.90,36
2,0.95,47


## Solution 5 — Multiple-testing sensitivity

In [42]:
rows = [{
    'method': 'raw',
    'alpha_or_q': 0.05,
    'discoveries': int((anova_p < 0.05).sum()),
}]

for method, alpha in [
    ('bonferroni', 0.05),
    ('holm', 0.05),
    ('fdr_bh', 0.01),
    ('fdr_bh', 0.05),
    ('fdr_bh', 0.10),
]:
    reject, _, _, _ = multipletests(anova_p, alpha=alpha, method=method)
    rows.append({
        'method': method,
        'alpha_or_q': alpha,
        'discoveries': int(reject.sum()),
    })

display(pd.DataFrame(rows))

,method,alpha_or_q,discoveries
0,raw,0.05,14479
1,bonferroni,0.05,5692
2,holm,0.05,5853
3,fdr_bh,0.01,10924
4,fdr_bh,0.05,13484
5,fdr_bh,0.10,15176


## Solution 7 — PCA dimension vs clustering quality

In [43]:
cluster_rows = []

for k in [2, 5, 10, 20, 40]:
    k_eff = min(k, n - 1)
    Z = PCA(n_components=k_eff).fit_transform(StandardScaler().fit_transform(X))
    pred = KMeans(
        n_clusters=y.nunique(), n_init=50, random_state=RANDOM_STATE
    ).fit_predict(Z)

    cluster_rows.append({
        'n_components': k_eff,
        'ARI': adjusted_rand_score(y, pred),
        'NMI': normalized_mutual_info_score(y, pred),
    })

cluster_sensitivity_df = pd.DataFrame(cluster_rows)
display(cluster_sensitivity_df)

,n_components,ARI,NMI
0,2,0.172418,0.381853
1,5,0.438195,0.656221
2,10,0.438195,0.656221
3,20,0.445143,0.659125
4,40,0.421609,0.645225


In [44]:
source = ColumnDataSource(cluster_sensitivity_df)
p_cluster_sensitivity = figure(
    height=380,
    width=850,
    title='Clustering quality vs PCA dimension',
    x_axis_label='Number of PCA components',
    y_axis_label='Adjusted Rand Index',
)
p_cluster_sensitivity.line('n_components', 'ARI', source=source, line_width=2)
p_cluster_sensitivity.scatter('n_components', 'ARI', source=source, size=8)
show(p_cluster_sensitivity)

# Part XVII — Further experiments

1. **Stability selection**: bootstrap and estimate gene-selection probabilities.
2. **Sparse PLS / PLS-DA**: compare supervised latent-variable methods with PCA/SIR.
3. **Ledoit–Wolf covariance shrinkage**: compare with empirical covariance.
4. **Sparse covariance / graphical models**: study conditional gene relationships.
5. **Permutation testing**: build empirical null distributions by shuffling labels.
6. **Random projections**: compare Johnson–Lindenstrauss projections with PCA.
7. **Debiased LASSO**: explore high-dimensional confidence intervals.
8. **Calibration**: evaluate probability calibration under nested resampling.
9. **External validation**: the most important next step for genuine biomarker claims.
10. **Selection stability vs predictive stability**: compare two models that predict similarly but choose different genes.

# Final synthesis

This case study demonstrates why high-dimensional statistics is more than “machine learning with many columns.”

When \(p\gg n\):

- OLS becomes non-identifiable.
- covariance estimation becomes unstable or singular.
- distance-based methods suffer from geometry changes.
- variable selection itself can overfit.
- regularization becomes essential.
- low-rank structure can make estimation feasible.
- sparse prediction and scientific inference are different tasks.
- thousands of p-values require multiplicity control.
- clustering and classification answer different questions.
- model assumptions matter enough that sometimes the correct action is **not to fit a method at all**.

The unifying idea is

\[
\boxed{\text{High dimension becomes tractable by exploiting structure.}}
\]

Common structures include

\[
\text{sparsity},\quad
\text{low rank},\quad
\text{regularity},\quad
\text{tree partitions},\quad
\text{latent states},\quad
\text{controlled error rates}.
\]

# Sources

- Kaggle — Leukemia gene expression – CuMiDa:  
  https://www.kaggle.com/datasets/brunogrisci/leukemia-gene-expression-cumida

- Feltes, B. C., Chandelier, E. B., Grisci, B. I., & Dorn, M. (2019).  
  *CuMiDa: An Extensively Curated Microarray Database for Benchmarking and Testing of Machine Learning Approaches in Cancer Research.*  
  Journal of Computational Biology, 26(4), 376–386.

- Grisci, B. I., Feltes, B. C., & Dorn, M. (2019).  
  *Neuroevolution as a tool for microarray gene expression pattern identification in cancer research.*  
  Journal of Biomedical Informatics, 89, 122–133.

- James, G., Witten, D., Hastie, T., Tibshirani, R., & Taylor, J.  
  *An Introduction to Statistical Learning.*

- Hastie, T., Tibshirani, R., & Friedman, J.  
  *The Elements of Statistical Learning.*